# Model B – From Text to Numbers: Word Embeddings with Gensim

Machine learning algorithms don't understand raw text — they work exclusively with
numerical data. In this stage we convert the pre‑processed `cleaned_text` column
(subject + body, fully cleaned) into **dense numerical feature vectors** using
word embeddings.

We'll use **Gensim's Word2Vec model**, which learns to represent words as
continuous vectors that capture semantic meaning — words used in similar contexts
end up with similar vector representations. To create a single vector for each
email, we average the word vectors of all words in the message.

This gives us a lightweight, semantically‑aware numerical representation ready
for classification, without the complexity or computational cost of deep
transformer models.

## 1. Load the Pre‑processed Data
We start from the cleaned dataset saved by the `data preprocessing` notebook.

**Note:** we will focus on the `cleaned_text` column since it doesn't have noisy data such as !, @, ...etc

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "preprocessed" / "processed_data.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} emails")
print(f"Label distribution:\n{df['label'].value_counts()}")
df[['cleaned_text', 'label']].head(2)

Loaded 164467 emails
Label distribution:
label
1    85646
0    78821
Name: count, dtype: int64


,cleaned_text,label
0,agree loser buck troubles caused small dimensi...,1
1,befriend jenna jameson upgrade sex pleasures t...,1


## 2. Tokenize the Text
`Word2Vec` expects sentences as lists of tokens. We split the cleaned text
into words — this is all the preprocessing Gensim needs.
In NLP, tokenization refers to the process of breaking a document or body of text into smaller units, known as tokens.

**Note:** we will split on whitespave since the data is already cleaned in preprocessing. 

In [2]:
df['tokens'] = df['cleaned_text'].apply(lambda x: x.split())
print("Sample tokens:", df['tokens'].iloc[0][:15])

Sample tokens: ['agree', 'loser', 'buck', 'troubles', 'caused', 'small', 'dimension', 'soon', 'lover', 'woman', 'able', 'resist', 'http', 'whitedone', 'com']


`Remark:` each email will be splited into a list of words and stored in the `tokens` list.

## 3. Train the Word2Vec Model
We train a Word2Vec model directly on our email corpus using Gensim.
This gives us word vectors that are tailored to the phishing/legitimate
language in our dataset.

Key hyperparameters:
- `vector_size`: dimensionality of the word vectors (150 is a good trade‑off)
- `window`: how many context words to consider on each side
- `min_count`: ignore words that appear fewer than this many times
- `workers`: use all CPU cores for faster training
- `sg`: 1 for Skip‑gram (usually better for smaller datasets), 0 for CBOW

#### The "Rare Word" Advantage (Crucial for Security)
In phishing detection, the most important signals often come from rare words or specific terminology (e.g., "cryptocurrency," "metamask," "unauthorized," or specific brand names).

`CBOW` tends to "smooth out" the data by averaging the context, which means it often ignores rare words in favor of more frequent, common words.

`Skip-gram` treats every word-context pair as a new observation. This allows it to learn high-quality representations even for words that don't appear very often in our 160k emails.

---
`Remark:` In Word2Vec, vocabulary refers to the unique set of all distinct words the model has learned from your dataset after filtering out rare terms and noise

In [7]:
import joblib
from pathlib import Path
from gensim.models import Word2Vec

model_dir = PROJECT_ROOT / "models"
model_path = model_dir / "word2vec_150d.joblib"

if model_path.exists():
    print(f"Model already exists in {model_path}, loading...")
    w2v_model = joblib.load(model_path)
else:
    print("No pre‑trained model found. Training a new Word2Vec model...")
    model_dir.mkdir(parents=True, exist_ok=True)          # create folder if needed
    
    w2v_model = Word2Vec(
        sentences=df['tokens'].tolist(),
        vector_size=150,
        window=5,
        min_count=2,
        workers=4,
        sg=1,
        epochs=30,
        seed=42
    )
    joblib.dump(w2v_model, model_path)
    print(f"Model trained and saved to {model_path}")

print(f"Vocabulary size: {len(w2v_model.wv)}")
if 'free' in w2v_model.wv:
    print(f"Vector for 'free' (first 10 dims): {w2v_model.wv['free'][:10]}...")

Model already exists in C:\Work\current\phishing_detection\models\word2vec_150d.joblib, loading...
Vocabulary size: 467821
Vector for 'free' (first 10 dims): [-0.623112    0.15396735 -0.29850122 -0.05686789 -0.08606317 -0.30855164
  0.11258993  0.3385862   0.15979289  0.54976153]...


## 4. Convert Each Email to a Single Vector
Emails are not fixed‑size objects, so we need to convert them into a fixed‑size vector representation. To get a fixed‑size vector for each email, we average the word vectors
of all its tokens. Words not in the vocabulary are simply skipped.

This is sometimes called **AvgWord2Vec** — a simple but effective way
to create document‑level representations from word embeddings.

In [4]:
def document_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X = np.stack(df['tokens'].apply(lambda t: document_vector(t, w2v_model)).values)
y = df['label'].values

print(f"Feature matrix shape: {X.shape}")

Feature matrix shape: (164467, 150)


## 5. Save the Word2Vec Model to disk
We persist the trained embedding model so the model‑training notebook can
load it directly — no need to retrain. So, by this way we will save both time and power.

Using `joblib` keeps the serialization simple and compatible with the
rest of the scikit‑learn / joblib ecosystem.

In [6]:
import joblib
from pathlib import Path

model_dir = PROJECT_ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)

# Save the Gensim Word2Vec model
w2v_path = model_dir / "word2vec_150d.joblib"
joblib.dump(w2v_model, w2v_path)

print(f"Word2Vec model saved to: {w2v_path}")

Word2Vec model saved to: C:\Work\current\phishing_detection\models\word2vec_150d.joblib


## 6. Save the Numerical Feature Matrix
To completely separate the embedding stage from model training, we persist the
numerical representation of every email as a CSV file in `data/processed/`.

The saved file contains:
- 150 feature columns (one per Word2Vec dimension)
- A `label` column (0/1)

The next notebook can load it with a single `pd.read_csv()` call.

In [9]:
feature_df = pd.DataFrame(
    X,
    columns=[f"dim_{i}" for i in range(X.shape[1])]
)
feature_df["label"] = y

processed_dir = PROJECT_ROOT / "data" / "model_b" / "vectorized_data"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "word2vec_features.csv"
feature_df.to_csv(output_path, index=False)

print(f"Feature matrix saved to: {output_path}")
print(f"Shape: {feature_df.shape}")
print("Columns (first 5):", feature_df.columns[:5].tolist())

Feature matrix saved to: C:\Work\current\phishing_detection\data\model_b\vectorized_data\word2vec_features.csv
Shape: (164467, 151)
Columns (first 5): ['dim_0', 'dim_1', 'dim_2', 'dim_3', 'dim_4']
